# 📘 Notebook 15: Stacks & Queues

### Course: *From Zero to Data Structures & Algorithms in Python*
**Instructor:** Ioannis Tsioulis

*Module D: Data Structures · Notebook 1 of 4 in this module · (15 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Explain what a **data structure** is, and the difference between its **interface** and its **implementation**
- Describe a **stack** (last in, first out) and implement it as a class
- Use a stack to solve real problems: reversing, matching brackets, undo
- Describe a **queue** (first in, first out) and explain why a plain list is a poor queue
- Implement an efficient queue with `collections.deque`
- State the Big O of every stack and queue operation

> **Prerequisites:** Notebooks 07 (classes) and 11 (Big O).
>
> 🧱 **Welcome to Module D.** So far we have asked *how* to process data. From now on we ask *how to store it*, because, as the set-versus-list experiment of Notebook 11 showed, the choice of container can change the speed of a program by a factor of hundreds.

## 1. What is a data structure?

### Intuition first
A kitchen keeps knives in a block, plates in a pile and spices on a rack. The same objects thrown into one big box would still all be there, but cooking would be painfully slow. **How things are organised determines how quickly you can use them.**

### Formal definition
A **data structure** is a way of organising data together with the **operations** allowed on it. Two aspects must be kept apart:

- the **interface**: *what* operations exist and what they do (the buttons on the outside);
- the **implementation**: *how* the data is actually stored to make those operations work (the machinery inside).

A user of the structure needs only the interface. This separation is exactly what a **class** provides, which is why every structure in this module is written as a class.

We begin with the two simplest structures. Both hold a sequence of items and both are deliberately **restrictive**: they let you add and remove items only at particular ends. That restriction is their strength.

## 2. The stack: last in, first out

### Intuition first
Think of a pile of plates. You put a new plate **on top**, and you take a plate **from the top**. The plate you added most recently is the first one you remove. Nobody pulls a plate out of the middle.

### Formal definition
A **stack** is a collection with two main operations, both acting on the same end, called the **top**:

| Operation | Meaning |
|---|---|
| `push(item)` | put an item on the top |
| `pop()` | remove and return the item on the top |
| `peek()` | look at the top item without removing it |
| `is_empty()` | is the stack empty? |

This behaviour is called **LIFO**: *Last In, First Out*.

### A Python list is already a good stack
The end of a list is the top. `append` pushes and `pop` pops, and Notebook 11 told us that both are **O(1)**.

In [ ]:
stack = []

stack.append("plate 1")      # push
stack.append("plate 2")
stack.append("plate 3")
print("Stack:", stack)

top = stack.pop()            # pop
print("Popped:", top)
print("Stack:", stack)
print("Top is now:", stack[-1])    # peek

### A `Stack` class
Why write a class if a list already works? Because a bare list allows **everything**: inserting in the middle, deleting from the bottom, sorting. A class exposes only the stack operations, so the structure cannot be misused, and the code that uses it says exactly what it means.

In [ ]:
class Stack:
    def __init__(self):
        self.items = []

    def push(self, item):
        self.items.append(item)

    def pop(self):
        if self.is_empty():
            return None
        return self.items.pop()

    def peek(self):
        if self.is_empty():
            return None
        return self.items[-1]

    def is_empty(self):
        return len(self.items) == 0

    def size(self):
        return len(self.items)

    def __str__(self):
        return f"Stack({self.items})  <- top"

stack = Stack()
stack.push(10)
stack.push(20)
stack.push(30)
print(stack)
print("Pop:", stack.pop())
print("Peek:", stack.peek())
print("Size:", stack.size())
print(stack)

> 🧠 This class is the `Library` pattern of Notebook 07: an object that **wraps a list** and offers a small, well-chosen set of methods. Every operation is **O(1)**.

## 3. What stacks are good for

A stack is the right tool whenever the most recent item must be handled first.

### Application 1: reversing
Push all the items, then pop them all. They come out in the opposite order.

In [ ]:
def reverse_text(text):
    stack = Stack()
    for char in text:
        stack.push(char)

    result = ""
    while not stack.is_empty():
        result += stack.pop()
    return result

print(reverse_text("Python"))

### Application 2: balanced brackets
Every code editor checks that brackets match. `(a + b) * (c - d)` is balanced. `(a + b))` and `((a + b)` are not. A stack solves this elegantly:

- on an **opening** bracket, push it;
- on a **closing** bracket, pop one. If there is nothing to pop, there is a closing bracket too many;
- at the end the stack must be **empty**. Anything left is an opening bracket that was never closed.

In [ ]:
def is_balanced(text):
    stack = Stack()
    for char in text:
        if char == "(":
            stack.push(char)
        elif char == ")":
            if stack.is_empty():
                return False          # a ")" with no matching "("
            stack.pop()
    return stack.is_empty()           # nothing may be left open

for expression in ["(a + b) * (c - d)", "((a + b)", "(a + b))", "())("]:
    print(f"{expression:<20} {is_balanced(expression)}")

### Application 3: undo
Each action is pushed on a stack. *Undo* pops the most recent one. The "back" button of a browser works the same way.

In [ ]:
document = ""
history = Stack()

for word in ["Hello", " world", "!!!"]:
    history.push(document)        # remember the state before the change
    document += word
    print("Typed:", repr(document))

document = history.pop()          # undo
print("Undo: ", repr(document))
document = history.pop()          # undo again
print("Undo: ", repr(document))

> 🔭 You have already met the most important stack of all. In Notebook 12, the unfinished recursive calls waited on the **call stack**: the last function called is the first to finish. LIFO again.

## 4. The queue: first in, first out

### Intuition first
A queue at a bakery. New customers join at the **back**, and the customer at the **front** is served next. Whoever arrived first is served first. That is simply fairness.

### Formal definition
A **queue** adds at one end and removes from the **other**:

| Operation | Meaning |
|---|---|
| `enqueue(item)` | add an item at the back |
| `dequeue()` | remove and return the item at the front |
| `peek()` | look at the front item without removing it |
| `is_empty()` | is the queue empty? |

This behaviour is called **FIFO**: *First In, First Out*.

### A first attempt with a list
`append` adds at the back, and `pop(0)` removes from the front.

In [ ]:
queue = []

queue.append("Anna")         # enqueue
queue.append("Kostas")
queue.append("Eleni")
print("Queue:", queue)

served = queue.pop(0)        # dequeue from the front
print("Served:", served)
print("Queue:", queue)

It works. But recall the table in Notebook 11: `pop(0)` is **O(n)**, because after the first element is removed, every remaining element must shift one place to the left. For a long queue this is a serious cost.

### The right tool: `collections.deque`
Python provides a structure built for this purpose. A **deque** (pronounced "deck", short for *double-ended queue*) adds and removes at **both** ends in **O(1)**. Its `popleft()` method removes from the front.

Let us measure the difference by emptying a queue of 100,000 items both ways.

In [ ]:
import time
from collections import deque

n = 100_000

queue_list = list(range(n))
start = time.perf_counter()
while queue_list:
    queue_list.pop(0)
list_time = time.perf_counter() - start

queue_deque = deque(range(n))
start = time.perf_counter()
while queue_deque:
    queue_deque.popleft()
deque_time = time.perf_counter() - start

print(f"list.pop(0):     {list_time:.4f} seconds")
print(f"deque.popleft(): {deque_time:.4f} seconds")
print(f"The deque was about {list_time / deque_time:,.0f} times faster.")

`while queue_list:` uses a convenient Python rule: an empty container counts as `False`, a non-empty one as `True`.

### A `Queue` class
The interface is what matters to the user. Inside, we choose the efficient implementation.

In [ ]:
class Queue:
    def __init__(self):
        self.items = deque()

    def enqueue(self, item):
        self.items.append(item)

    def dequeue(self):
        if self.is_empty():
            return None
        return self.items.popleft()

    def peek(self):
        if self.is_empty():
            return None
        return self.items[0]

    def is_empty(self):
        return len(self.items) == 0

    def size(self):
        return len(self.items)

    def __str__(self):
        return f"front -> {list(self.items)} <- back"

queue = Queue()
queue.enqueue("Anna")
queue.enqueue("Kostas")
queue.enqueue("Eleni")
print(queue)
print("Dequeue:", queue.dequeue())
print(queue)

> 🧠 Notice what just happened. We changed the **implementation** from a list to a deque, and the **interface** (`enqueue`, `dequeue`) stayed the same. Any code that uses the `Queue` class would keep working, only faster. That is the benefit of separating the two.

## 5. What queues are good for

A queue is the right tool whenever things must be handled **in the order they arrive**: print jobs, customers, network packets, tasks waiting for a processor.

### A worked example: a printer
Documents are sent to a printer faster than it can print them. They wait in a queue and are printed strictly in order of arrival.

In [ ]:
printer = Queue()

for document in ["report.pdf", "photo.png", "invoice.docx"]:
    printer.enqueue(document)
    print(f"Queued: {document}")

print()
while not printer.is_empty():
    print(f"Printing {printer.dequeue()}... ({printer.size()} left in the queue)")

### A worked example: hot potato
Children stand in a circle and pass a potato. After a fixed number of passes, whoever holds it leaves the circle. The last child remaining wins. A queue models the circle: passing the potato means moving the person at the front to the back.

In [ ]:
def hot_potato(names, passes):
    circle = Queue()
    for name in names:
        circle.enqueue(name)

    while circle.size() > 1:
        for i in range(passes):
            circle.enqueue(circle.dequeue())     # front goes to the back
        print("Out:", circle.dequeue())

    return circle.dequeue()

winner = hot_potato(["Anna", "Kostas", "Eleni", "Dimitris", "Maria"], 3)
print("Winner:", winner)

## 6. Stack or queue?

| | Stack | Queue |
|---|---|---|
| Order | **LIFO**: last in, first out | **FIFO**: first in, first out |
| Add | `push`, at the top | `enqueue`, at the back |
| Remove | `pop`, from the top | `dequeue`, from the front |
| Picture | a pile of plates | a line at a bakery |
| Typical uses | undo, brackets, recursion, backtracking | scheduling, printing, serving requests in order |
| Cost of every operation | O(1) | O(1) with a deque |

---
## ✏️ Exercises

### Exercise 1 (Predict the output)
Without running it, work out what the code below prints. Then check.

In [ ]:
s = Stack()
s.push(1)
s.push(2)
s.push(3)
s.pop()
s.push(4)
s.pop()
s.pop()
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
s = Stack()
s.push(1)
s.push(2)
s.push(3)
s.pop()          # removes 3
s.push(4)
s.pop()          # removes 4
s.pop()          # removes 2
print(s)         # only 1 is left
```
</details>

### Exercise 2 (Reverse a list)
Write a function `reverse_list(items)` that returns a new list with the elements in reverse order, using a `Stack`.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def reverse_list(items):
    stack = Stack()
    for item in items:
        stack.push(item)
    result = []
    while not stack.is_empty():
        result.append(stack.pop())
    return result

print(reverse_list([1, 2, 3, 4, 5]))
```
</details>

### Exercise 3 (All kinds of brackets)
Extend `is_balanced` so that it handles round `()`, square `[]` and curly `{}` brackets. Each closing bracket must match the **most recent** unclosed opening bracket, so `([)]` is **not** balanced. (Hint: a dictionary that maps each closing bracket to its opening partner keeps the code short.)

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def is_balanced_all(text):
    pairs = {")": "(", "]": "[", "}": "{"}
    stack = Stack()
    for char in text:
        if char in "([{":
            stack.push(char)
        elif char in pairs:
            if stack.pop() != pairs[char]:
                return False
    return stack.is_empty()

for expression in ["{[()]}", "([)]", "((]", "print(data[0]['x'])"]:
    print(f"{expression:<22} {is_balanced_all(expression)}")
```

*If the stack is empty, `pop()` returns `None`, which is not equal to any bracket, so the function correctly returns `False`.*
</details>

### Exercise 4 (Browser history)
Simulate the back button of a browser. Visit the pages `"home"`, `"news"`, `"sport"` and `"weather"` in that order, pushing each page you **leave** onto a stack. Then press *back* twice and print the page you end up on.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
history = Stack()
current_page = "home"

for page in ["news", "sport", "weather"]:
    history.push(current_page)
    current_page = page
    print("Visiting:", current_page)

for i in range(2):
    current_page = history.pop()
    print("Back to: ", current_page)
```
</details>

### Exercise 5 (Supermarket checkout)
Five customers join a queue: `"Anna"`, `"Kostas"`, `"Eleni"`, `"Dimitris"`, `"Maria"`. Serve them in order. After the second customer has been served, a new customer `"Nikos"` joins. Print each customer as they are served.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
checkout = Queue()
for name in ["Anna", "Kostas", "Eleni", "Dimitris", "Maria"]:
    checkout.enqueue(name)

served = 0
while not checkout.is_empty():
    print("Serving:", checkout.dequeue())
    served += 1
    if served == 2:
        checkout.enqueue("Nikos")
```
</details>

### Exercise 6 (Is it a palindrome?)
Check whether a word is a palindrome using **both** structures. Put every letter into a stack and into a queue. Then repeatedly compare what `pop` returns (the last letter) with what `dequeue` returns (the first letter).

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def is_palindrome(word):
    stack = Stack()
    queue = Queue()
    for char in word:
        stack.push(char)
        queue.enqueue(char)

    while not stack.is_empty():
        if stack.pop() != queue.dequeue():
            return False
    return True

print(is_palindrome("level"))
print(is_palindrome("python"))
```
</details>

### Exercise 7 (Evaluate a postfix expression)
In **postfix** notation the operator comes after its two operands: `3 4 +` means `3 + 4`, and `3 4 + 2 *` means `(3 + 4) * 2`. No brackets are ever needed. Write `evaluate_postfix(expression)`: read the tokens from left to right; push each number; on an operator, pop two numbers, apply it, and push the result. (Hint: `expression.split()` gives the tokens. The **first** number popped is the right-hand operand.)

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def evaluate_postfix(expression):
    stack = Stack()
    for token in expression.split():
        if token in "+-*/":
            right = stack.pop()
            left = stack.pop()
            if token == "+":
                stack.push(left + right)
            elif token == "-":
                stack.push(left - right)
            elif token == "*":
                stack.push(left * right)
            else:
                stack.push(left / right)
        else:
            stack.push(float(token))
    return stack.pop()

print(evaluate_postfix("3 4 +"))
print(evaluate_postfix("3 4 + 2 *"))
print(evaluate_postfix("10 2 8 * + 3 -"))
```

*This is how many calculators and programming-language interpreters actually evaluate arithmetic.*
</details>

### Exercise 8 (A queue from two stacks, a little harder)
Build a class `QueueFromStacks` with `enqueue` and `dequeue`, using **only two `Stack` objects** for storage. (Hint: push new items onto the first stack. To dequeue, if the second stack is empty, pop everything from the first stack onto the second, which reverses the order, then pop from the second.)

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
class QueueFromStacks:
    def __init__(self):
        self.inbox = Stack()
        self.outbox = Stack()

    def enqueue(self, item):
        self.inbox.push(item)

    def dequeue(self):
        if self.outbox.is_empty():
            while not self.inbox.is_empty():
                self.outbox.push(self.inbox.pop())
        return self.outbox.pop()

q = QueueFromStacks()
q.enqueue("a")
q.enqueue("b")
q.enqueue("c")
print(q.dequeue(), q.dequeue())
q.enqueue("d")
print(q.dequeue(), q.dequeue())
```

*Reversing a stack twice restores the original order. This is a classic interview question, and a nice demonstration that an interface can be built on top of a completely different structure.*
</details>

## 🔑 Key takeaways

- A **data structure** is a way of organising data plus the operations allowed on it. Keep its **interface** apart from its **implementation**.
- A **stack** is **LIFO**: `push` and `pop` act on the top. A Python list is a good stack, since `append` and `pop` are O(1).
- Stacks handle "most recent first" problems: reversing, matching brackets, undo, the call stack of recursion.
- A **queue** is **FIFO**: `enqueue` at the back, `dequeue` from the front.
- `list.pop(0)` is O(n), so a list is a poor queue. `collections.deque` gives O(1) at both ends.
- A restricted structure is not a weaker one: the restriction is what makes it clear, safe and fast.

---
**Next:** *Notebook 16: Linked Lists*, a structure made of nodes that point to each other.

---
*© Ioannis Tsioulis. *From Zero to Data Structures & Algorithms in Python*. Prepared for educational use.*